# Link Prediction mit GINE

Gleiche Daten, gleicher Trainingsloop (`gnn_pipeline/training.py`), gleiche Eingangsschicht (`EdgeAwareInput`),
gleicher Decoder (`LinkDecoder`) und gleiche Hyperparameter wie in `GATV2.ipynb`; nur das Message Passing ist
`GINEConv` statt `GATv2Conv`. Die Breite (128) ist in beiden Modellen gleich, die Parameterzahlen liegen in
derselben Größenordnung.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from gnn_pipeline import (
    load_bundle, set_seed, EdgeAwareInput, LinkDecoder, batch_norm, count_parameters,
    train_link_predictor, evaluate_link_predictor, run_seeds, plot_learning_curves, auc_by_link_type,
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

bundle = load_bundle(device=device)   # processed/philadelphia_split.pt
data, train_data, val_data, test_data = bundle.data, bundle.train, bundle.val, bundle.test

in_channels = data.x.size(1)
edge_dim = data.edge_attr.size(1)

## Graph Isomorphism Network mit Kantenfeatures (GINE)

GIN aggregiert Nachbarn per **Summe** und wendet dann ein MLP an – das ist so ausdrucksstark wie
der Weisfeiler-Lehman-Test. GINE (Hu et al., *Strategies for Pre-training GNNs*) erweitert dies um
Kantenfeatures:

$$x_i' = \mathrm{MLP}\Big((1+\epsilon)\, x_i + \sum_{j \in \mathcal{N}(i)} \mathrm{ReLU}(x_j + W_e\, e_{ji})\Big)$$

Über `edge_dim` projiziert `GINEConv` die Kantenfeatures linear auf die Knotendimension, damit
die Addition $x_j + W_e e_{ji}$ möglich ist.

**Unterschiede zu GATv2, die für die Interpretation wichtig sind:**
- keine Attention, alle Nachbarn zählen gleich (gewichtet nur durch die Kantenfeatures),
- Summen-Aggregation macht GINE sensibel für den Knotengrad – bei nur `x, y` als Knotenfeature
  ist das Eingangssignal dünn; `node_features=["coords", "node_type", "degree"]` in
  `data_prep.ipynb` ist ein naheliegendes Experiment,
- `BatchNorm` zwischen den Schichten stabilisiert die Summen. Sie läuft ohne Running-Stats
  (`batch_norm` in `gnn_pipeline/models.py`), weil Training und Auswertung denselben Graphen sehen.

In [ ]:
from torch_geometric.nn import GINEConv

def _mlp(in_dim, out_dim):
    return nn.Sequential(
        nn.Linear(in_dim, out_dim),
        batch_norm(out_dim),
        nn.ReLU(),
        nn.Linear(out_dim, out_dim),
    )

class GINELinkPredictor(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, edge_dim, dropout=0.3, train_eps=True):
        super().__init__()
        # gemeinsame Eingangsschicht (identisch zu GATv2): Koordinaten + Summe der eingehenden Kantenfeatures
        self.input = EdgeAwareInput(in_channels, edge_dim, hidden_channels)
        self.conv1 = GINEConv(_mlp(hidden_channels, hidden_channels), train_eps=train_eps, edge_dim=edge_dim)
        self.conv2 = GINEConv(_mlp(hidden_channels, hidden_channels), train_eps=train_eps, edge_dim=edge_dim)
        self.conv3 = GINEConv(_mlp(hidden_channels, out_channels), train_eps=train_eps, edge_dim=edge_dim)
        self.bn0 = batch_norm(hidden_channels)
        self.bn1 = batch_norm(hidden_channels)
        self.bn2 = batch_norm(hidden_channels)
        self.dropout = nn.Dropout(dropout)
        self.decoder = LinkDecoder(out_channels, hidden_channels, dropout)   # identisch zu GATv2

    def forward(self, x, edge_index, edge_attr):
        x = self.bn0(self.input(x, edge_index, edge_attr)).relu()
        x = self.dropout(self.bn1(self.conv1(x, edge_index, edge_attr)).relu())
        x = self.dropout(self.bn2(self.conv2(x, edge_index, edge_attr)).relu())
        return self.conv3(x, edge_index, edge_attr)

    def predict_link(self, node_embeddings, edge_label_index):
        return self.decoder(node_embeddings, edge_label_index)

In [ ]:
MODEL_NAME = "GINE"
# Identisch in GINE.ipynb und GATV2.ipynb, damit sich die Modelle nur im Encoder unterscheiden
HIDDEN, OUT, DROPOUT = 128, 128, 0.3
EPOCHS, LR, WEIGHT_DECAY = 300, 0.002, 5e-4
SEED = bundle.meta.get("seed", 42)

def make_model():
    return GINELinkPredictor(in_channels, HIDDEN, OUT, edge_dim, dropout=DROPOUT)

set_seed(SEED)
model = make_model().to(device)
print(model)
print(f"Parameter: {count_parameters(model):,}")

In [ ]:
TRAIN_KWARGS = dict(epochs=EPOCHS, lr=LR, weight_decay=WEIGHT_DECAY)

history = train_link_predictor(model, train_data, val_data, test_data, log_every=10, **TRAIN_KWARGS)

# model hat jetzt die Gewichte der Epoche mit der besten Val-AUC
test_loss, test_auc = evaluate_link_predictor(model, test_data)
print(f"Test (Modell der besten Val-Epoche): Loss {test_loss:.4f} | ROC-AUC {test_auc:.4f}")

## Learning Curve

`plot_learning_curves` (in `gnn_pipeline/training.py`) zeigt die komplette `history`: links den BCE-Loss,
rechts die ROC-AUC, jeweils für Train, Val und Test. Alle drei Kurven sind im eval-Modus (ohne Dropout) auf
**demselben** Message-Passing-Graphen gemessen; keine der bewerteten Kanten liegt selbst im Graphen. Die
Kurven sind damit direkt vergleichbar. Die graue Linie markiert die Epoche mit der besten Val-AUC; auf diese
Gewichte wird das Modell nach dem Training zurückgesetzt, alle folgenden Zellen nutzen also dieses Modell.

**Worauf man achten sollte:**
- **Train über Val/Test:** Ein kleiner, stabiler Abstand ist normal (das Modell hat die Trainingspaare
  gesehen). Wächst er, während die Val-AUC stagniert, ist das Overfitting – das Checkpointing fängt es ab.
- **Val und Test liegen aufeinander:** erwartet, beide Splits sind gleich gezogen und sehen denselben Graphen.
- **Einordnung:** Die Heuristik-Baselines in `data_prep.ipynb` (Luftlinie ≈ 0.72, Grad ≈ 0.53) sind die
  Messlatte. Alles darüber hat das Modell aus der Graphstruktur gelernt.
- **Einzelner Lauf:** Eine Kurve ist eine Stichprobe. Für Aussagen über das Modell zählt die
  Mehr-Seed-Auswertung unten.

In [ ]:
plot_learning_curves(history, title=f"{MODEL_NAME}: Learning Curve ({len(history['val_auc'])} Epochen, Full-Batch, Seed {SEED})")
plt.show()

## Mehr-Seed-Auswertung

Einzelne Läufe streuen (Initialisierung, Dropout, nicht-deterministische GPU-Kernels), frühere
Läufe mit identischem Setup lagen bis zu ~0.015 AUC auseinander. Für den Vergleich GINE ↔ GATv2 zählt daher
Mittelwert ± Standardabweichung über mehrere Seeds, jeweils an der Epoche mit der besten Val-AUC (Auswahl auf
Val, nicht auf Test). Die Ergebnisse werden nach `processed/seeds_<Modell>.csv` geschrieben.

Laufzeit: pro Seed etwa so lange wie das Training oben.

In [ ]:
SEEDS = [0, 1, 2, 3, 4]
seed_df, seed_histories = run_seeds(make_model, train_data, val_data, test_data, SEEDS, **TRAIN_KWARGS)
seed_df.to_csv(f"processed/seeds_{MODEL_NAME}.csv", index=False)
seed_df

## Ablation: Tragen die Kantenfeatures etwas bei?

Gleiches Modell, gleiche Seeds, aber alle Kantenfeatures auf 0 gesetzt. Liegt die Differenz zur
Mehr-Seed-Auswertung oben innerhalb der Standardabweichung, nutzt das Modell die Kantenfeatures praktisch
nicht. Standardmäßig aus (`RUN_ABLATION = False`), weil es die Laufzeit der Mehr-Seed-Auswertung verdoppelt.

In [ ]:
RUN_ABLATION = False

def without_edge_features(split):
    split = split.clone()
    split.edge_attr = torch.zeros_like(split.edge_attr)
    return split

if RUN_ABLATION:
    ablation_df, _ = run_seeds(make_model, *(without_edge_features(s) for s in (train_data, val_data, test_data)),
                               SEEDS, **TRAIN_KWARGS)
    display(pd.DataFrame({"mit Kantenfeatures": seed_df[["val_auc", "test_auc"]].agg(["mean", "std"]).stack(),
                          "ohne Kantenfeatures": ablation_df[["val_auc", "test_auc"]].agg(["mean", "std"]).stack()}))

## Visualisierung der Knoten-Embeddings

PCA der gelernten Embeddings (Modell der besten Val-Epoche, vollständiger Graph) auf 2 Dimensionen,
eingefärbt nach Knotentyp (Zone / Kreuzung laut TNTP-Dateikopf).

In [ ]:
from sklearn.decomposition import PCA

model.eval()
with torch.no_grad():
    emb = model(data.x, data.edge_index, data.edge_attr).cpu().numpy()

emb_2d = PCA(n_components=2).fit_transform(emb)
plot_df = pd.DataFrame({'component_1': emb_2d[:, 0], 'component_2': emb_2d[:, 1], 'node_type': bundle.node_type})

plt.figure(figsize=(12, 10))
sns.scatterplot(data=plot_df, x='component_1', y='component_2', hue='node_type', palette='viridis', alpha=0.7, s=20)
plt.title(f'2D PCA of {MODEL_NAME} Node Embeddings (colored by node type)', size=16)
plt.xlabel('Principal Component 1'); plt.ylabel('Principal Component 2')
plt.legend(title='Node Type'); plt.grid(True, linestyle='--', alpha=0.6); plt.tight_layout()
plt.show()

## Fehleranalyse: Welche Straßentypen sagt das Modell gut vorher?

Für jeden Link-Typ: ROC-AUC der positiven Val-Kanten dieses Typs gegen alle negativen Val-Paare, dazu die
mittlere vorhergesagte Wahrscheinlichkeit. Typ 7 mit Kapazität 999999 sind die künstlichen
Zonen-Konnektoren, keine echten Straßen.

(Die frühere Korrelation „Kantenfeature ↔ Label“ war konstruktionsbedingt ≈ 0, weil Negative zufällig
gezogene Kantenfeatures tragen, und wurde deshalb ersetzt.)

In [ ]:
link_df = auc_by_link_type(model, val_data, bundle.edge_attr_columns)
display(link_df)

plt.figure(figsize=(10, 5))
sns.barplot(x=link_df.index, y=link_df["ROC-AUC vs. alle Negativen"], color="#2a78d6")
plt.ylim(max(0.0, link_df["ROC-AUC vs. alle Negativen"].min() - 0.05), 1.0)
plt.title(f"{MODEL_NAME}: Val-ROC-AUC nach Link-Typ der positiven Kante", size=14)
plt.xlabel("Link-Typ"); plt.ylabel("ROC-AUC"); plt.grid(axis="y", linestyle="--", alpha=0.6); plt.tight_layout()
plt.show()